# SQL Refresher 1 b

## imports

In [1]:
import math
import numpy as np
import pandas as pd

import psycopg2

## my_select_query_pandas() - function to run a select query and return rows in a Pandas dataframe

In [2]:
#
# function to run a select query and return rows in a pandas dataframe
# pandas puts all numeric values from postgres to float
# if it will fit in an integer, change it to integer
#

def my_select_query_pandas(query, rollback_before_flag, rollback_after_flag):
    "function to run a select query and return rows in a pandas dataframe"
    
    if rollback_before_flag:
        connection.rollback()
    
    df = pd.read_sql_query(query, connection)
    
    if rollback_after_flag:
        connection.rollback()
    
    # fix the float columns that really should be integers
    
    for column in df:
    
        if df[column].dtype == "float64":

            fraction_flag = False

            for value in df[column].values:
                
                if not np.isnan(value):
                    if value - math.floor(value) != 0:
                        fraction_flag = True

            if not fraction_flag:
                df[column] = df[column].astype('Int64')
    
    return(df)
    

## Connect to the Postgres database

In [3]:
connection = psycopg2.connect(
    user = "postgres",
    password = "ucb",
    host = "postgres",
    port = "5432",
    database = "postgres"
)

## Create a cursor for the connection

In [4]:
cursor = connection.cursor()

# Lab: SQL - Set Operations

## Create a table from the results of a query 

In [5]:
connection.rollback()

query = """

drop table if exists temp_zip_1;

drop table if exists temp_zip_2;

create table temp_zip_1
as
select zip, city, state, population
from zip_codes
where state = 'CA' and (city = 'Berkeley' or city = 'Los Angeles')
;

create table temp_zip_2
as
select zip, city, state, population
from zip_codes
where state = 'CA' and (city = 'Los Angeles')
;

"""

cursor.execute(query)

connection.commit()

## Verify the first table we just created

In [6]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select *
from temp_zip_1
order by state, city, zip

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,zip,city,state,population
0,94702,Berkeley,CA,17092
1,94703,Berkeley,CA,21937
2,94704,Berkeley,CA,29190
3,94705,Berkeley,CA,13365
4,94707,Berkeley,CA,11916
...,...,...,...,...
66,90067,Los Angeles,CA,2314
67,90068,Los Angeles,CA,20982
68,90073,Los Angeles,CA,916
69,90077,Los Angeles,CA,8993


## Verify the second table we just created

In [7]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select *
from temp_zip_2
order by state, city, zip

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,zip,city,state,population
0,90001,Los Angeles,CA,58975
1,90002,Los Angeles,CA,53111
2,90003,Los Angeles,CA,72741
3,90004,Los Angeles,CA,61586
4,90005,Los Angeles,CA,39479
...,...,...,...,...
57,90067,Los Angeles,CA,2314
58,90068,Los Angeles,CA,20982
59,90073,Los Angeles,CA,916
60,90077,Los Angeles,CA,8993


## union combines rows from two queries, removing duplicates

In [8]:
# Union will rm dups
rollback_before_flag = True
rollback_after_flag = True

query = """

(select *
from temp_zip_1)
union
(select *
from temp_zip_2)
order by state, city, zip

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,zip,city,state,population
0,94702,Berkeley,CA,17092
1,94703,Berkeley,CA,21937
2,94704,Berkeley,CA,29190
3,94705,Berkeley,CA,13365
4,94707,Berkeley,CA,11916
...,...,...,...,...
66,90067,Los Angeles,CA,2314
67,90068,Los Angeles,CA,20982
68,90073,Los Angeles,CA,916
69,90077,Los Angeles,CA,8993


## "union all" combines rows from two queries, keeps duplicates

In [9]:
rollback_before_flag = True
rollback_after_flag = True

query = """

(select *
from temp_zip_1)
union all
(select *
from temp_zip_2)
order by state, city, zip

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,zip,city,state,population
0,94702,Berkeley,CA,17092
1,94703,Berkeley,CA,21937
2,94704,Berkeley,CA,29190
3,94705,Berkeley,CA,13365
4,94707,Berkeley,CA,11916
...,...,...,...,...
128,90073,Los Angeles,CA,916
129,90077,Los Angeles,CA,8993
130,90077,Los Angeles,CA,8993
131,90089,Los Angeles,CA,3862


## intersect returns rows that are in both queries

In [10]:
rollback_before_flag = True
rollback_after_flag = True

query = """

(select *
from temp_zip_1)
intersect
(select *
from temp_zip_2)
order by state, city, zip

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,zip,city,state,population
0,90001,Los Angeles,CA,58975
1,90002,Los Angeles,CA,53111
2,90003,Los Angeles,CA,72741
3,90004,Los Angeles,CA,61586
4,90005,Los Angeles,CA,39479
...,...,...,...,...
57,90067,Los Angeles,CA,2314
58,90068,Los Angeles,CA,20982
59,90073,Los Angeles,CA,916
60,90077,Los Angeles,CA,8993


## except (minus) returns rows in the first query that are not in the second query

In [11]:
rollback_before_flag = True
rollback_after_flag = True

# Note: Most SQL databases support minus.  
# Postgres calls minus except. 
# Set operations tend to be somewhat vendor dependant

query = """

(select *
from temp_zip_1)
except
(select *
from temp_zip_2)
order by state, city, zip

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,zip,city,state,population
0,94702,Berkeley,CA,17092
1,94703,Berkeley,CA,21937
2,94704,Berkeley,CA,29190
3,94705,Berkeley,CA,13365
4,94707,Berkeley,CA,11916
5,94708,Berkeley,CA,11455
6,94709,Berkeley,CA,11740
7,94710,Berkeley,CA,7461
8,94720,Berkeley,CA,2971


## drop our two tables

In [12]:
connection.rollback()

query = """

drop table if exists temp_zip_1;

drop table if exists temp_zip_2;

"""

cursor.execute(query)

connection.commit()

## You try it - using the states table, create temp_states_1 that contains state names that start with a C or a T, create temp_states_2 that contains state names that start with a T, demonstrate set operations, drop the temp tables when you are done

In [13]:
# Check the stats table
rollback_before_flag = True
rollback_after_flag = True

query = """

select *
from states
LIMIT 5
"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,state,state_name,latitude,longitude,population,area,density
0,PA,Pennsylvania,40.9479,-77.6280,12790950,43851.7864,291.69
1,OH,Ohio,40.1758,-82.6609,11639989,40511.7523,287.32
2,TN,Tennessee,35.8113,-85.9375,6644470,41397.0517,160.51
3,MS,Mississippi,32.6082,-89.7987,2988710,45241.4545,66.06
4,CT,Connecticut,41.5278,-72.7300,3581504,4791.4988,747.47


In [14]:
connection.rollback()

query = """

drop table if exists temp_state_1;

drop table if exists temp_state_2;

create table temp_state_1
as
select *
from states
where state like 'C%' or state like 'T%';

create table temp_state_2
as
select *
from states
where state like 'T%'
;

"""

cursor.execute(query)

connection.commit()

In [15]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select *
from temp_state_1

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,state,state_name,latitude,longitude,population,area,density
0,TN,Tennessee,35.8113,-85.9375,6644470,41397.0517,160.51
1,CT,Connecticut,41.5278,-72.7300,3581504,4791.4988,747.47
2,TX,Texas,31.1946,-100.1082,27883996,233570.0589,119.38
3,CO,Colorado,38.9961,-105.5937,5531101,101924.2851,54.27
4,CA,California,37.2519,-119.2326,39140087,97473.6922,401.55


In [16]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select *
from temp_state_2

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,state,state_name,latitude,longitude,population,area,density
0,TN,Tennessee,35.8113,-85.9375,6644470,41397.0517,160.51
1,TX,Texas,31.1946,-100.1082,27883996,233570.0589,119.38


In [17]:
# Use set operation to Check the except
rollback_before_flag = True
rollback_after_flag = True

# Note: Most SQL databases support minus.  
# Postgres calls minus except. 
# Set operations tend to be somewhat vendor dependant

query = """

(select *
from temp_state_1)
except
(select *
from temp_state_2)
order by state
"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,state,state_name,latitude,longitude,population,area,density
0,CA,California,37.2519,-119.2326,39140087,97473.6922,401.55
1,CO,Colorado,38.9961,-105.5937,5531101,101924.2851,54.27
2,CT,Connecticut,41.5278,-72.7300,3581504,4791.4988,747.47


In [18]:
# Drop the temp tables
connection.rollback()

query = """

drop table if exists temp_state_1;

drop table if exists temp_state_2;

"""

cursor.execute(query)

connection.commit()

# Lab: SQL - Join Operations

## Joins combine columns from two or more tables; link the primary key in the parent table to the foreign key in the child table; this is an inner join where parent rows without child rows are not included; here we join the stores table to the sales table

In [19]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select s.city, sum(sa.total_amount) as total_sales
from stores as s 
     join sales as sa 
         on s.store_id = sa.store_id
group by s.city
order by total_sales desc

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,city,total_sales
0,Berkeley,25041060
1,Seattle,22024512
2,Dallas,19408260
3,Miami,17692404
4,Nashville,14573172


## Join stores to sales to line_items, this would need to use two cols (composite)

In [20]:
rollback_before_flag = True
rollback_after_flag = True

# This would tell us the best selling items in each city
query = """

select s.city, l.product_id, sum(quantity) as total_quantity
from stores as s 
     join sales as sa 
         on s.store_id = sa.store_id
     join line_items as l
         on sa.store_id = l.store_id and sa.sale_id = l.sale_id
group by s.city, l.product_id
order by 1, 3 desc

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,city,product_id,total_quantity
0,Berkeley,1,464274
1,Berkeley,4,405637
2,Berkeley,6,346508
3,Berkeley,2,290858
4,Berkeley,8,232038
5,Berkeley,7,174252
6,Berkeley,3,115469
7,Berkeley,5,57719
8,Dallas,1,359615
9,Dallas,4,314383


## Join the stores to sales to line_items to products

In [21]:
# Joining stores-sales-line item-products, this would give us the name of the product
rollback_before_flag = True
rollback_after_flag = True

query = """

select s.city as store, p.description as product, sum(quantity) as total_quantity
from stores as s 
     join sales as sa 
         on s.store_id = sa.store_id
     join line_items as l
         on sa.store_id = l.store_id and sa.sale_id = l.sale_id 
     join products as p
         on l.product_id = p.product_id
group by s.city, p.description
order by 1, 3 desc

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,store,product,total_quantity
0,Berkeley,Pistachio Salmon,464274
1,Berkeley,Eggplant Lasagna,405637
2,Berkeley,Curry Chicken,346508
3,Berkeley,Teriyaki Chicken,290858
4,Berkeley,Brocolli Stir Fry,232038
5,Berkeley,Tilapia Piccata,174252
6,Berkeley,Spinach Orzo,115469
7,Berkeley,Chicken Salad,57719
8,Dallas,Pistachio Salmon,359615
9,Dallas,Eggplant Lasagna,314383


## Join customers to sales to line_items to products

In [22]:
# People who bought a lot of certain products
rollback_before_flag = True
rollback_after_flag = True

# Always check the ERD for how to join tables
# || is the string concatenator
query = """

select cu.first_name || cu.last_name as customer_name,
       p.description as product, 
       sum(quantity) as total_quantity
from customers as cu
     join sales as sa 
         on cu.customer_id = sa.customer_id
     join line_items as l
         on sa.store_id = l.store_id and sa.sale_id = l.sale_id
     join products as p
         on l.product_id = p.product_id
group by customer_name, product
order by 1, 3 desc

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,customer_name,product,total_quantity
0,AarenCossons,Pistachio Salmon,107
1,AarenCossons,Eggplant Lasagna,99
2,AarenCossons,Curry Chicken,92
3,AarenCossons,Teriyaki Chicken,68
4,AarenCossons,Brocolli Stir Fry,49
...,...,...,...
248152,ZuzanaLuckham,Teriyaki Chicken,48
248153,ZuzanaLuckham,Brocolli Stir Fry,39
248154,ZuzanaLuckham,Tilapia Piccata,32
248155,ZuzanaLuckham,Spinach Orzo,20


## There can be customers who have signed up and not bought anything yet; an inner join will not return them

In [23]:
rollback_before_flag = True
rollback_after_flag = True

# The inner join wont return peole who signed up but have not bought anything yet
query = """

select cu.first_name || ' ' || cu.last_name as customer_name,
       sa.sale_date
from customers as cu
     join sales as sa 
         on cu.customer_id = sa.customer_id
where cu.customer_id in (24521, 9318)
order by 1, 2

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,customer_name,sale_date
0,Lenci Hanaford,2020-01-04
1,Lenci Hanaford,2020-01-18
2,Lenci Hanaford,2020-02-02
3,Lenci Hanaford,2020-02-10
4,Lenci Hanaford,2020-02-18
5,Lenci Hanaford,2020-02-26
6,Lenci Hanaford,2020-02-28
7,Lenci Hanaford,2020-03-05
8,Lenci Hanaford,2020-03-07
9,Lenci Hanaford,2020-03-14


## left outer join will include parent rows without child rows;  this will include customers who have not bought anything yet

In [24]:
rollback_before_flag = True
rollback_after_flag = True

# Left join (left outer join) will return those peps
query = """

select cu.first_name || ' ' || cu.last_name as customer_name,
       sa.sale_date
from customers as cu
     left outer join sales as sa 
         on cu.customer_id = sa.customer_id
where cu.customer_id in (24521, 9318)
order by 1, 2

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,customer_name,sale_date
0,Lenci Hanaford,2020-01-04
1,Lenci Hanaford,2020-01-18
2,Lenci Hanaford,2020-02-02
3,Lenci Hanaford,2020-02-10
4,Lenci Hanaford,2020-02-18
5,Lenci Hanaford,2020-02-26
6,Lenci Hanaford,2020-02-28
7,Lenci Hanaford,2020-03-05
8,Lenci Hanaford,2020-03-07
9,Lenci Hanaford,2020-03-14


## left outer join must be used on all subsequent joins

In [25]:
rollback_before_flag = True
rollback_after_flag = True


query = """

select s.city as store, 
       cu.first_name || ' ' || cu.last_name as customer_name,
       p.description as product, 
       sum(quantity) as total_quantity
from stores as s 
     join customers as cu
         on s.store_id = cu.closest_store_id
     left outer join sales as sa 
         on s.store_id = sa.store_id and cu.customer_id = sa.customer_id
     left outer join line_items as l
         on sa.store_id = l.store_id and sa.sale_id = l.sale_id
     left outer join products as p
         on l.product_id = p.product_id
where cu.customer_id in (24521, 9318)
group by store, customer_name, product
order by 1, 3 desc

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,store,customer_name,product,total_quantity
0,Miami,Lenci Hanaford,Tilapia Piccata,25
1,Miami,Lenci Hanaford,Teriyaki Chicken,37
2,Miami,Lenci Hanaford,Spinach Orzo,8
3,Miami,Lenci Hanaford,Pistachio Salmon,64
4,Miami,Lenci Hanaford,Eggplant Lasagna,36
5,Miami,Lenci Hanaford,Curry Chicken,25
6,Miami,Lenci Hanaford,Chicken Salad,7
7,Miami,Lenci Hanaford,Brocolli Stir Fry,34
8,Seattle,Tracy Agott,None,<NA>


## You try it - For each store, for each day of the week, for each product, list the total number sold.  

Hints:  

extract(dow from sa.sale_date) will give you the day of week as an integer from 0 to 6 with 0 = Sunday.   You will find this helpful for ordering.

to_char(sa.sale_date, 'Day') will give you the day of week in the form "Sunday", "Monday", etc. You will find this more user friendly than the numeric dow.



# Lab: SQL - Type 1 Subqueries

## type 1 subqueries have no linkage to the outer query;  they can be pulled out and run standalone; can be used in a where clause

In [26]:
rollback_before_flag = True
rollback_after_flag = True

# This would give us customer who has never buy anything
query = """

select cu.last_name, cu.first_name, cu.customer_id
from customers as cu
where cu.customer_id not in (select distinct customer_id from sales)
order by 1, 2, 3
"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,last_name,first_name,customer_id
0,Agott,Tracy,9318
1,Arnke,Daniella,2676
2,Assandri,Hyacintha,21030
3,Borman,Felice,7600
4,Breit,Domini,30414
5,Butterick,Jacenta,5148
6,Camillo,Marysa,11113
7,Dukelow,Lilas,295
8,Dukesbury,Corinna,21644
9,Ellaway,Lorianna,10999


## Type 1 subqueries can also be used in place of table names in from clauses;  the comma is a cross product join (also called Cartesian Product);  match all rows of one table with all rows of another table; good for matching all rows to a total as shown

In [27]:
rollback_before_flag = True
rollback_after_flag = True

# Total projected sale of a product per year and it's percentage of sales
# Two type 1 subqueries
query = """

select a.product, 
       a.total_sales_for_product, 
       round((a.total_sales_for_product / b.total_sales) * 100, 1) as percentage_of_total_sales 
from 

    (
     select p.description as product, 
            sum(l.quantity) * 12 as total_sales_for_product
     from sales sa
          join line_items l
              on sa.store_id = l.store_id and sa.sale_id = l.sale_id
          join products p
              on l.product_id = p.product_id
      group by product
    ) as a,
    
    (
     select sum(sa.total_amount) as total_sales 
     from sales sa
    ) as b

order by 3 desc
;


"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,product,total_sales_for_product,percentage_of_total_sales
0,Pistachio Salmon,21945336,22.2
1,Eggplant Lasagna,19188696,19.4
2,Curry Chicken,16426608,16.6
3,Teriyaki Chicken,13740156,13.9
4,Brocolli Stir Fry,10967808,11.1
5,Tilapia Piccata,8246844,8.4
6,Spinach Orzo,5481228,5.6
7,Chicken Salad,2742732,2.8


## with clause is an alternative syntax;  most people find it much easier

In [28]:
rollback_before_flag = True
rollback_after_flag = True

# Better syntax that is more intuitive
query = """

with a as (
            select p.description as product, 
                sum(l.quantity) * 12 as total_sales_for_product
            from sales as sa
            join line_items as l
                on sa.store_id = l.store_id and sa.sale_id = l.sale_id
            join products as p
                on l.product_id = p.product_id
            group by product
          )
          ,
     b as (
            select sum(sa.total_amount) as total_sales 
            from sales as sa     
          )
select a.product, 
       a.total_sales_for_product, 
       round((a.total_sales_for_product / b.total_sales) * 100, 1) as percentage_of_total_sales 
from a, b
order by 3 desc
;


"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,product,total_sales_for_product,percentage_of_total_sales
0,Pistachio Salmon,21945336,22.2
1,Eggplant Lasagna,19188696,19.4
2,Curry Chicken,16426608,16.6
3,Teriyaki Chicken,13740156,13.9
4,Brocolli Stir Fry,10967808,11.1
5,Tilapia Piccata,8246844,8.4
6,Spinach Orzo,5481228,5.6
7,Chicken Salad,2742732,2.8


## Views are basically a "with" clause that is permanent and can be shared by all queries; view do not use storage - they are not the same as a "create table as select" that we saw earlier

In [29]:
connection.rollback()

query = """

drop view if exists v_total_sales;

create or replace view v_total_sales
as
select sum(sa.total_amount) as total_sales 
from sales as sa
;

"""

cursor.execute(query)

connection.commit()

## Views can be used just like tables in queries

In [30]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select *
from v_total_sales
;


"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,total_sales
0,98739408


## View to join stores to sales to customers; also creates several useful derived columns

In [31]:
connection.rollback()

query = """

drop view if exists v_join_s_sa_cu;

create view v_join_s_sa_cu
as
select s.store_id,
       s.street as store_street,
       s.city as store_city,
       s.state as store_state,
       s.zip as store_zip,
       s.latitude as store_latitude,
       s.longitude as store_longitude,
       sa.sale_id,
       sa.customer_id,
       sa.sale_date,
       extract(dow from sa.sale_date) as dow,
       to_char(sa.sale_date, 'Day') as day_of_week,
       extract(month from sa.sale_date) as month_number,
       to_char(sa.sale_date, 'Month') as month_name,
       sa.total_amount,
       cu.first_name,
       cu.last_name,
       (cu.first_name || ' ' || cu.last_name) as full_name,
       (cu.last_name || ', ' || cu.first_name) as last_name_first,
       cu.street as customer_street,
       cu.city as customer_city,
       cu.state as customer_state,
       cu.zip as customer_zip,
       cu.distance
from stores s
     join sales as sa
        on s.store_id = sa.store_id
     join customers as cu
        on sa.customer_id = cu.customer_id
;

"""

cursor.execute(query)

connection.commit()

## This join is huge! Remeber to use a limit, an aggregation, where clause, etc. to limit the return!

In [32]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select *
from v_join_s_sa_cu
limit 5
;


"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,store_id,store_street,store_city,store_state,store_zip,store_latitude,store_longitude,sale_id,customer_id,sale_date,...,total_amount,first_name,last_name,full_name,last_name_first,customer_street,customer_city,customer_state,customer_zip,distance
0,1,3000 Telegraph Ave,Berkeley,CA,94705,37.8555,-122.2604,1,710,2020-01-01,...,48,Felic,Wawer,Felic Wawer,"Wawer, Felic",2 Bunting Parkway,Berkeley,CA,94705,1
1,1,3000 Telegraph Ave,Berkeley,CA,94705,37.8555,-122.2604,2,1477,2020-01-01,...,132,Cody,Childerley,Cody Childerley,"Childerley, Cody",954 Brown Terrace,Oakland,CA,94610,3
2,1,3000 Telegraph Ave,Berkeley,CA,94705,37.8555,-122.2604,3,1820,2020-01-01,...,48,Eula,Deware,Eula Deware,"Deware, Eula",3 Crownhardt Road,Albany,CA,94706,3
3,1,3000 Telegraph Ave,Berkeley,CA,94705,37.8555,-122.2604,4,1272,2020-01-01,...,96,Jarrad,Brusle,Jarrad Brusle,"Brusle, Jarrad",42 Elmside Plaza,Berkeley,CA,94709,2
4,1,3000 Telegraph Ave,Berkeley,CA,94705,37.8555,-122.2604,5,1986,2020-01-01,...,36,Roley,Carvell,Roley Carvell,"Carvell, Roley",7 Center Terrace,Berkeley,CA,94707,3


## View to join stores to sales to customer to line_items to products

In [33]:
connection.rollback()

query = """

drop view if exists v_join_s_sa_cu_l_p;

create view v_join_s_sa_cu_l_p
as
select s.store_id,
       s.street as store_street,
       s.city as store_city,
       s.state as store_state,
       s.zip as store_zip,
       s.latitude as store_latitude,
       s.longitude as store_longitude,
       sa.sale_id,
       sa.customer_id,
       sa.sale_date,
       extract(dow from sa.sale_date) as dow,
       to_char(sa.sale_date, 'Day') as day_of_week,
       extract(month from sa.sale_date) as month_number,
       to_char(sa.sale_date, 'Month') as month_name,
       sa.total_amount,
       cu.first_name,
       cu.last_name,
       (cu.first_name || ' ' || cu.last_name) as full_name,
       (cu.last_name || ', ' || cu.first_name) as last_name_first,
       cu.street as customer_street,
       cu.city as customer_city,
       cu.state as customer_state,
       cu.zip as customer_zip,
       cu.distance,
       l.line_item_id,
       l.product_id,
       l.quantity,
       (l.quantity * 12) as line_item_amount,
       p.description as product
from stores s
     join sales as sa
        on s.store_id = sa.store_id
     join customers as cu
        on sa.customer_id = cu.customer_id
     join line_items as l
        on sa.store_id = l.store_id and sa.sale_id = l.sale_id 
     join products as p
        on l.product_id = p.product_id
;

"""

cursor.execute(query)

connection.commit()

## An even bigger join - be careful and use a limit, aggregation, where, etc.

In [34]:
rollback_before_flag = True
rollback_after_flag = True

# A very useful view
query = """

select *
from v_join_s_sa_cu_l_p
limit 5
;


"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,store_id,store_street,store_city,store_state,store_zip,store_latitude,store_longitude,sale_id,customer_id,sale_date,...,customer_street,customer_city,customer_state,customer_zip,distance,line_item_id,product_id,quantity,line_item_amount,product
0,1,3000 Telegraph Ave,Berkeley,CA,94705,37.8555,-122.2604,1,710,2020-01-01,...,2 Bunting Parkway,Berkeley,CA,94705,1,1,1,2,24,Pistachio Salmon
1,1,3000 Telegraph Ave,Berkeley,CA,94705,37.8555,-122.2604,1,710,2020-01-01,...,2 Bunting Parkway,Berkeley,CA,94705,1,2,2,2,24,Teriyaki Chicken
2,1,3000 Telegraph Ave,Berkeley,CA,94705,37.8555,-122.2604,2,1477,2020-01-01,...,954 Brown Terrace,Oakland,CA,94610,3,1,1,4,48,Pistachio Salmon
3,1,3000 Telegraph Ave,Berkeley,CA,94705,37.8555,-122.2604,2,1477,2020-01-01,...,954 Brown Terrace,Oakland,CA,94610,3,2,3,2,24,Spinach Orzo
4,1,3000 Telegraph Ave,Berkeley,CA,94705,37.8555,-122.2604,2,1477,2020-01-01,...,954 Brown Terrace,Oakland,CA,94610,3,3,4,1,12,Eggplant Lasagna


## You try it - Create a view called v_store_sales_by_day_of_week that finds each store's sales by day of week. Display the store id, city, day of week in numeric for sorting purposes, day of week in string for display purposes, and total_sales. Test the view by selecting from it.  You can drop the view when you are done with it if you want.

In [35]:
connection.rollback()

query = """

drop view if exists v_store_sales_by_day_of_week;

create view v_store_sales_by_day_of_week
as
select s.store_id,
       s.city,
       extract(dow from sa.sale_date) as dow,
       to_char(sa.sale_date, 'Day') as day_of_week,
       sum(sa.total_amount) as total_sales
from stores s
     join sales as sa
        on s.store_id = sa.store_id
group by s.store_id, s.city, dow, day_of_week
;

"""

cursor.execute(query)

connection.commit()

In [36]:
rollback_before_flag = True
rollback_after_flag = True

# Very useful biz intel
query = """

select *
from v_store_sales_by_day_of_week
order by city, dow
;


"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,store_id,city,dow,day_of_week,total_sales
0,1,Berkeley,0,Sunday,4694640
1,1,Berkeley,1,Monday,3340116
2,1,Berkeley,2,Tuesday,1752036
3,1,Berkeley,3,Wednesday,3546144
4,1,Berkeley,4,Thursday,3507660
5,1,Berkeley,5,Friday,3273240
6,1,Berkeley,6,Saturday,4927224
7,3,Dallas,0,Sunday,3650748
8,3,Dallas,1,Monday,2602980
9,3,Dallas,2,Tuesday,1352760


# Lab: SQL - Type 2 Subqueries

## Type 2 subqueries have a linkage to the outer query;   cannot be pulled out and run standalone; notorously slow and high memory usage - be careful! in the where clause, zip in the subquery is linked to cu.zip in the outer query

In [37]:
rollback_before_flag = True
rollback_after_flag = True

# Finding customers in zipcodes that has > 50000
# Doing one more join might be more human - readable
query = """

select cu.customer_id, 
       cu.first_name, 
       cu.last_name,
       sum(sa.total_amount) as total_sales
from customers cu
     join sales as sa
         on cu.customer_id = sa.customer_id
where (select population from zip_codes where zip = cu.zip) > 50000
group by cu.customer_id, cu.first_name, cu.last_name
;


"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,customer_id,first_name,last_name,total_sales
0,21025,Currey,Nobes,2076
1,17844,Rosy,Cockcroft,2448
2,3635,Anabella,Sibun,3516
3,24610,Theo,Truter,2304
4,19635,Lewiss,MacLaig,2328
...,...,...,...,...
3497,18686,Noble,Carek,2316
3498,3771,Martie,Masey,2724
3499,20882,Artemis,Trodd,1884
3500,10564,Angie,Strettle,6408


In [38]:
rollback_before_flag = True
rollback_after_flag = True

# Finding customers in zipcodes that has > 50000
# Doing one more join might be more human - readable
query = """

SELECT
  cu.customer_id,
  cu.first_name,
  cu.last_name,
  cu.zip            AS zip_code,
  z.population,
  SUM(sa.total_amount) AS total_sales
FROM customers cu
JOIN sales sa
  ON cu.customer_id = sa.customer_id
JOIN zip_codes z
  ON z.zip = cu.zip
WHERE
  z.population > 50000
GROUP BY
  cu.customer_id,
  cu.first_name,
  cu.last_name,
  cu.zip,
  z.population;
;


"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,customer_id,first_name,last_name,zip_code,population,total_sales
0,3449,Whitby,Mennithorp,94501,63843,2076
1,3450,Royall,Kaas,94501,63843,4224
2,3451,Sula,Kneller,94501,63843,2784
3,3452,Casper,Burford,94501,63843,4260
4,3453,Leighton,Huxley,94501,63843,3408
...,...,...,...,...,...,...
3497,31072,Aime,Gaunson,37066,50328,3048
3498,31073,Frank,Eagleston,37129,58040,3048
3499,31074,Huntington,Ballach,37129,58040,3624
3500,31075,Mattie,Pentycost,37129,58040,3564


## Type 2 subqueries can also go in the having clause on aggregations

In [39]:
rollback_before_flag = True
rollback_after_flag = True


# Gets customers that buys more than avg
query = """

select cu.customer_id, 
       cu.first_name, 
       cu.last_name,
       sum(sa.total_amount) as total_sales
from customers cu
     join sales as sa
         on cu.customer_id = sa.customer_id
where cu.zip = '94720'
group by cu.customer_id, cu.first_name, cu.last_name
having sum(sa.total_amount) >
        (select avg(sa2.total_amount)
         from stores as s2
              join sales as sa2
                  on s2.store_id = sa2.store_id
         where s2.store_id = cu.closest_store_id)
;


"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,customer_id,first_name,last_name,total_sales
0,762,Jeffrey,Connechy,4080
1,763,Dorthy,Filby,2472
2,764,Rasia,Millwall,4896
3,765,Antoni,Fulloway,1608
4,766,Cybill,Watchorn,4104
5,767,Christie,Blaszczak,4500
6,768,Vallie,Astlet,3696
7,769,Doyle,Zaczek,3156
8,770,Dwayne,Boyd,4128
9,771,Natalya,Cossor,4512


## You try it - using a type 2 subquery: find customers in Alameda, CA whose total sales is greater than the average sales for their zip code.  Display the customer_id, first_name, last_name, and total_sales. (this may take a while to run)

In [40]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select cu.customer_id, 
       cu.first_name, 
       cu.last_name,
       sum(sa.total_amount) as total_sales
from customers cu
     join sales as sa
         on cu.customer_id = sa.customer_id
where cu.city = 'Alameda' and cu.state = 'CA'
group by cu.customer_id, cu.first_name, cu.last_name
having sum(sa.total_amount) >
        (select avg(sa2.total_amount)
         from customers as cu2
              join sales as sa2
                  on cu2.customer_id = sa2.customer_id
         where cu.zip = cu2.zip)
;


"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,customer_id,first_name,last_name,total_sales
0,3449,Whitby,Mennithorp,2076
1,3450,Royall,Kaas,4224
2,3451,Sula,Kneller,2784
3,3452,Casper,Burford,4260
4,3453,Leighton,Huxley,3408
...,...,...,...,...
298,4552,Krisha,Hovenden,2652
299,4553,Flori,Marchington,3144
300,4554,Rock,Boundey,3312
301,4555,Nana,Bartholin,3108
